# Parameter sweep: payload mass against the sizing verdicts

This notebook runs the T-gantry's pick-and-place cycle for a range of payload masses with the Python module `kinematics_sim` and shows how peak and RMS motor torque, the thermal utilisation and the verdicts change. Install the module first (see `docs/python.md`), plus `pandas` and `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kinematics_sim as ks

pd.DataFrame(ks.machines())

The default config of the machine, and the value the sweep changes (kg):

In [ ]:
base = ks.Config("t-gantry")
base["mass.payload_kg"]

One analysis and one set of verdicts per payload mass. Each call runs the cycle to its second pass and measures it; the calls release the GIL, so a thread pool runs them on all processors.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

masses = np.linspace(0.5, 8.0, 8)

def judge(mass):
    cfg = base.copy()
    cfg["mass.payload_kg"] = float(mass)
    load = pd.DataFrame(ks.analyze(cfg))
    checks = pd.DataFrame(ks.verdicts(cfg))
    load["payload_kg"] = mass
    load["all_checks_pass"] = bool(checks["ok"].all())
    load["failing"] = ", ".join(sorted(set(checks.loc[~checks["ok"], "check"])))
    return load

with ThreadPoolExecutor() as pool:
    sweep = pd.concat(pool.map(judge, masses), ignore_index=True)
sweep[["payload_kg", "motor", "peak_torque_nm", "rms_torque_nm", "thermal_utilisation", "all_checks_pass", "failing"]]

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
for motor, rows in sweep.groupby("motor"):
    a.plot(rows.payload_kg, rows.peak_torque_nm, "o-", label=f"motor {motor} peak")
    a.plot(rows.payload_kg, rows.rms_torque_nm, "s--", label=f"motor {motor} RMS")
    b.plot(rows.payload_kg, rows.thermal_utilisation, "o-", label=f"motor {motor}")
a.set(xlabel="payload [kg]", ylabel="torque [N m]", title="Motor torque over one cycle")
b.axhline(1.0, color="grey", lw=1)
b.set(xlabel="payload [kg]", ylabel="RMS / rated torque", title="Thermal utilisation")
a.legend(); b.legend(); plt.tight_layout()

The heaviest payload of the sweep for which every check of the config still passes:

In [ ]:
passing = sweep.groupby("payload_kg")["all_checks_pass"].all()
passing[passing].index.max() if passing.any() else None

A single run's signals as a DataFrame, e.g. to plot the following errors:

In [ ]:
run = pd.DataFrame(ks.run(base, cycle=True, time=1.0))
run.plot(x="time", y=[c for c in run.columns if c.startswith("axis.") and c.endswith("following_error")], figsize=(8, 3))